In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 1.Setup MLflow

In [ ]:
!pip install dagshub
!pip install mlflow

In [ ]:
import dagshub
dagshub.init(repo_owner='vasiliDandurishvili', repo_name='ML_Fraud_Detection', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

In [ ]:
# ============================================================
# SETUP: Libraries + MLflow + DagsHub
# ============================================================
import os
import gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin

# DagsHub + MLflow (OAuth-ით)
import dagshub
dagshub.init(repo_owner='vasiliDandurishvili', repo_name='ML_Fraud_Detection', mlflow=True)

import mlflow
import mlflow.sklearn

# ექსპერიმენტის შექმნა
mlflow.set_experiment("RandomForest_Training")



## 2. Data Loading and merge

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================
DATA_PATH = '/kaggle/input/datasets/vasilidandurishvili/dataset/'

print("Loading data...")
train_tx = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_id = pd.read_csv(DATA_PATH + 'train_identity.csv')

# Merge transaction + identity (LEFT JOIN — identity არ აქვს ყველა transaction-ს)
train = train_tx.merge(train_id, on='TransactionID', how='left')

# Memory-ის გათავისუფლება
del train_tx, train_id
gc.collect()

print(f"Train shape: {train.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f}")
print(f"Memory: {train.memory_usage().sum() / 1024**2:.1f} MB")

# 1. Cleaning

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# თითოეული სვეტის NaN %
nan_pct = (train.isna().sum() / len(train)).sort_values(ascending=False)

# Plot 1: Histogram - რამდენი სვეტი რა NaN %-ის რეგიონშია
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(nan_pct.values, bins=50, edgecolor='black', color='steelblue')
axes[0].set_xlabel('NaN პროცენტი')
axes[0].set_ylabel('სვეტების რაოდენობა')
axes[0].set_title('NaN-ის % distribution სვეტებში')
axes[0].axvline(x=0.5, color='orange', linestyle='--', label='50%')
axes[0].axvline(x=0.7, color='red', linestyle='--', label='70%')
axes[0].axvline(x=0.9, color='darkred', linestyle='--', label='90%')
axes[0].legend()

# Plot 2: ყოველი threshold-ისთვის რამდენი სვეტი წაიშლება
thresholds = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
dropped_counts = [(nan_pct > t).sum() for t in thresholds]

axes[1].bar([f'{int(t*100)}%' for t in thresholds], dropped_counts, color='coral', edgecolor='black')
axes[1].set_xlabel('NaN threshold')
axes[1].set_ylabel('წასაშლელი სვეტების რაოდენობა')
axes[1].set_title('რამდენ სვეტს წავშლით სხვადასხვა threshold-ზე?')

for i, v in enumerate(dropped_counts):
    axes[1].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nThreshold | Dropped | Remaining")
print("-" * 35)
for t, d in zip(thresholds, dropped_counts):
    print(f"  {int(t*100)}%    |   {d:3d}   |   {train.shape[1] - d}")

In [ ]:
# Top 30 ყველაზე NaN-იანი სვეტი
top_nan = nan_pct.head(30)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_nan)), top_nan.values, color='steelblue', edgecolor='black')
plt.yticks(range(len(top_nan)), top_nan.index)
plt.xlabel('NaN პროცენტი')
plt.title('Top 30 ყველაზე NaN-იანი სვეტი')
plt.axvline(x=0.5, color='orange', linestyle='--', alpha=0.7, label='50%')
plt.axvline(x=0.9, color='red', linestyle='--', alpha=0.7, label='90%')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# RUNS: RF_Cleaning_Threshold_X (3 ცალცალკე run)
# ============================================================
thresholds_to_test = [0.5, 0.75, 0.9]
cleaning_results = {}

for threshold in thresholds_to_test:
    with mlflow.start_run(run_name=f"RF_Cleaning_Threshold_{int(threshold*100)}"):
        
        high_nan_cols = nan_pct[nan_pct > threshold].index.tolist()
        train_cleaned_test = train.drop(columns=high_nan_cols)
        
        # MLflow log
        mlflow.log_param("nan_threshold", threshold)
        mlflow.log_param("strategy", "drop_high_nan_columns")
        mlflow.log_metric("dropped_features", len(high_nan_cols))
        mlflow.log_metric("remaining_features", train_cleaned_test.shape[1])
        mlflow.log_metric("dropped_pct", len(high_nan_cols) / train.shape[1] * 100)
        
        cleaning_results[threshold] = {
            'dropped': len(high_nan_cols),
            'remaining': train_cleaned_test.shape[1]
        }
        
        print(f"Threshold {int(threshold*100)}%: წაიშალა {len(high_nan_cols):3d}, დარჩა {train_cleaned_test.shape[1]}")

print("\n" + "="*50)
print("Summary:")
for t, r in cleaning_results.items():
    print(f"  {int(t*100)}% threshold → დარჩა {r['remaining']} feature")

In [ ]:
# საბოლოო threshold-ის არჩევა
FINAL_THRESHOLD = 0.50   

with mlflow.start_run(run_name="RF_Cleaning_Final"):
    
    high_nan_cols = nan_pct[nan_pct > FINAL_THRESHOLD].index.tolist()
    train_cleaned = train.drop(columns=high_nan_cols)
    
    mlflow.log_param("final_nan_threshold", FINAL_THRESHOLD)
    mlflow.log_param("reasoning", "kept_max_features_minimal_loss")
    mlflow.log_metric("dropped_features", len(high_nan_cols))
    mlflow.log_metric("remaining_features", train_cleaned.shape[1])
    mlflow.log_metric("rows", train_cleaned.shape[0])
    
    print(f" საბოლოო Cleaning: წაიშალა {len(high_nan_cols)}, დარჩა {train_cleaned.shape[1]} feature")
    print(f" train_cleaned ცვლადი მზადაა Feature Engineering-ისთვის")

In [ ]:
# დამატებითი შემოწმება
print(f"train shape (უცვლელი): {train.shape}")
print(f"train_cleaned shape: {train_cleaned.shape}")
print(f"სხვაობა: {train.shape[1] - train_cleaned.shape[1]} სვეტი წაიშალა")

# 2. Feature Engineering

ვქმნით ახალ features-ებს არსებული მონაცემებიდან:
- **TransactionAmt_log** — log transform skewness-ის გამოსასწორებლად
- **TransactionAmt_decimal** — decimal ნაწილი (round numbers fraud-ის სიგნალია)
- **hour, day** — TransactionDT-დან დროითი features
- **P/R_emaildomain_bin** — email domain გამარტივება (gmail.com, gmail.co.uk → gmail)

In [ ]:
import matplotlib.pyplot as plt

# ვფილტრავთ 5000-მდე უკეთესი ვიზუალიზაციისთვის
amt_filtered = train_cleaned[train_cleaned['TransactionAmt'] <= 5000]['TransactionAmt']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Original distribution (5000-მდე)
axes[0].hist(amt_filtered, bins=100, color='steelblue', edgecolor='black')
axes[0].set_title('TransactionAmt (≤ 5000) - skewed!')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')

# Log transformed (სრულზე — log-ის შემდეგ outlier-ები არ მოვა პრობლემა)
axes[1].hist(np.log1p(train_cleaned['TransactionAmt']), bins=100, color='coral', edgecolor='black')
axes[1].set_title('log(TransactionAmt + 1) - more normal')
axes[1].set_xlabel('log(Amount)')

plt.tight_layout()
plt.show()

print(f"Original skewness: {train_cleaned['TransactionAmt'].skew():.2f}")
print(f"Log-transformed skewness: {np.log1p(train_cleaned['TransactionAmt']).skew():.2f}")

# Range-ების მიხედვით ტრანზაქციების რაოდენობა
print("\n" + "="*50)
print("ტრანზაქციების განაწილება Amount-ის range-ებში:")
print("="*50)

ranges = [
    (0, 100),
    (100, 500),
    (500, 1000),
    (1000, 2000),
    (2000, 5000),
    (5000, 10000),
    (10000, float('inf'))
]

total = len(train_cleaned)
for low, high in ranges:
    if high == float('inf'):
        count = (train_cleaned['TransactionAmt'] > low).sum()
        label = f">{low}"
    else:
        count = ((train_cleaned['TransactionAmt'] > low) & 
                 (train_cleaned['TransactionAmt'] <= high)).sum()
        label = f"{low}-{high}"
    pct = count / total * 100
    print(f"  {label:>15s}:  {count:>7,} ({pct:5.2f}%)")

print(f"\nსულ ტრანზაქცია:  {total:,}")
print(f"მაქსიმალური თანხა: ${train_cleaned['TransactionAmt'].max():.2f}")
print(f"საშუალო თანხა: ${train_cleaned['TransactionAmt'].mean():.2f}")
print(f"მედიანა: ${train_cleaned['TransactionAmt'].median():.2f}")

In [ ]:
# ============================================================
# RUN: RF_Feature_Engineering
# ============================================================
with mlflow.start_run(run_name="RF_Feature_Engineering"):
    
    df = train_cleaned.copy()
    new_features = []
    
    # ─────────────────────────────────────────────────────────
    # 1. TransactionAmt_log
    # ─────────────────────────────────────────────────────────
    # რატომ: TransactionAmt ძალიან skewed-ია (ბევრი პატარა, ცოტა უზარმაზარი).
    # log transform გადაიყვანს უფრო ნორმალურ distribution-ად, რაც
    # კრიტიკულია Logistic Regression-ისთვის — ის წრფივი მოდელია და
    # outlier-ები coefficient-ებს ანახვევენ. log(x+1) ვიყენებთ რომ
    # log(0) = -infinity პრობლემა თავიდან ავიცილოთ.
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    new_features.append('TransactionAmt_log')
    
    # ─────────────────────────────────────────────────────────
    # 2. TransactionAmt_decimal
    # ─────────────────────────────────────────────────────────
    # რატომ: fraud-ი ხშირად "მრგვალი" თანხებით ხდება (მაგ. $100.00, $500.00),
    # რეალური მომხმარებლები კი ხშირად ხარჯავენ "ლუწ" თანხებს ($23.47).
    # decimal == 0 → შესაძლო fraud სიგნალი. ეს patterns-ი მოდელს ვერ
    # გამოეჩხრიკება უბრალო TransactionAmt-დან.
    df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
    new_features.append('TransactionAmt_decimal')
    
    # ─────────────────────────────────────────────────────────
    # 3. hour
    # ─────────────────────────────────────────────────────────
    # რატომ: TransactionDT უბრალოდ წამებია reference moment-დან —
    # მოდელისთვის უაზრო რიცხვი. ჩვენ კი ვიცით რომ fraud უფრო ხშირია
    # ღამის საათებზე (რეალური მომხმარებლები სძინავთ). hour feature-ი
    # ამ patterns-ს მოდელს მისცემს. (TransactionDT / 3600) % 24 →
    # 0-24 დიაპაზონი.
    df['hour'] = (df['TransactionDT'] / 3600) % 24
    new_features.append('hour')
    
    # ─────────────────────────────────────────────────────────
    # 4. day
    # ─────────────────────────────────────────────────────────
    # რატომ: კვირის დღეც ფაქტორია — შაბათ-კვირას fraud pattern-ი
    # შეიძლება განსხვავდებოდეს სამუშაო დღეებისგან. (TransactionDT /
    # 86400) % 7 → 0-7 (კვირის დღე reference-დან).
    df['day'] = (df['TransactionDT'] / (3600 * 24)) % 7
    new_features.append('day')
    
    # ─────────────────────────────────────────────────────────
    # 5. P_emaildomain_bin & R_emaildomain_bin
    # ─────────────────────────────────────────────────────────
    # რატომ: original email domain-ში ბევრი ვარიაცია გვაქვს —
    # gmail.com, gmail.co.uk, gmail.fr ცალცალკე category-ებად
    # ფიქსირდებიან, მაშინ როცა არსებითად ერთიდაიგივეა. ვამოვლებთ
    # მხოლოდ პირველ ნაწილს (gmail-ი ყველა მათგანში). ეს ამცირებს
    # cardinality-ს და მოდელს უფრო სტაბილურ სიგნალს აძლევს.
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in df.columns:
            df[col + '_bin'] = df[col].astype(str).str.split('.').str[0]
            new_features.append(col + '_bin')
    
    # ----- MLflow logging -----
    mlflow.log_param("new_features_count", len(new_features))
    mlflow.log_param("new_features", str(new_features))
    mlflow.log_param("transformations", "log, decimal, time_extract, email_simplify")
    mlflow.log_metric("features_before_FE", train_cleaned.shape[1])
    mlflow.log_metric("features_after_FE", df.shape[1])
    mlflow.log_metric("added_features", len(new_features))
    
    print(f"შექმნილია {len(new_features)} ახალი feature:")
    for f in new_features:
        print(f"  - {f}")
    print(f"\nShape ცვლილება: {train_cleaned.shape} → {df.shape}")
    
    train_fe = df

print(f"\n train_fe ცვლადი მზადაა Feature Selection-ისთვის")

In [ ]:
print("ახალი features-ის სტატისტიკა:\n")
print(train_fe[['TransactionAmt', 'TransactionAmt_log', 'TransactionAmt_decimal', 
                 'hour', 'day']].describe())

print("\n\nFraud rate hour-ის მიხედვით:")
hour_fraud = train_fe.groupby(train_fe['hour'].astype(int))['isFraud'].mean()
print(hour_fraud)

plt.figure(figsize=(12, 4))
hour_fraud.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Fraud Rate by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Fraud Rate')
plt.axhline(y=train_fe['isFraud'].mean(), color='red', linestyle='--', label='Average fraud rate')
plt.legend()
plt.tight_layout()
plt.show()

# 3. Feature Selection

ვტესტავთ სხვადასხვა მიდგომას feature-ების გასაფილტრად:
- **Low Variance** — სვეტები რომლებიც თითქმის უცვლელია
- **High Correlation** — ერთმანეთთან მაღალი კორელაციის სვეტები (multicollinearity)
- **Target Correlation** — isFraud-თან ნული-კორელაციის სვეტები

თითოეული მიდგომა ცალკე MLflow run-ში დაილოგება, საბოლოოდ კი შევაერთებთ.

In [ ]:
# იდენტიფიკატორი + target
TARGET = 'isFraud'
ID_COL = 'TransactionID'

# კონკურსის ოფიციალური categorical სვეტები
cat_cols_official = [
    'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo',
    'P_emaildomain_bin', 'R_emaildomain_bin'
]
cat_cols_official += [f'id_{i:02d}' for i in range(12, 39)]

# ვტოვებთ მხოლოდ იმ სვეტებს რომელიც არსებობს train_fe-ში
cat_cols = [c for c in cat_cols_official if c in train_fe.columns]

# numerical = ყველა დანარჩენი (გარდა target-ისა, ID-ისა და categorical-ისა)
num_cols = [c for c in train_fe.columns if c not in cat_cols + [TARGET, ID_COL]]

print(f"Categorical features: {len(cat_cols)}")
print(f"Numerical features: {len(num_cols)}")
print(f"სულ: {len(cat_cols) + len(num_cols)}")

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# ============================================================
# RUN: RF_FS_LowVariance
# ============================================================
with mlflow.start_run(run_name="RF_FS_LowVariance"):
    
    # NaN-ის გარეშე variance ვერ ვითვლით — ვავსებთ median-ით ანალიზისთვის
    num_data_filled = train_fe[num_cols].fillna(train_fe[num_cols].median())
    
    
    variance_threshold = 0.01
    selector = VarianceThreshold(threshold=variance_threshold)
    selector.fit(num_data_filled)
    
    # რომელი სვეტები არის low-variance
    low_var_mask = ~selector.get_support()
    low_var_cols = [num_cols[i] for i, m in enumerate(low_var_mask) if m]
    
    print(f"Low variance სვეტები (variance < {variance_threshold}): {len(low_var_cols)}")
    print(f"მაგალითები: {low_var_cols[:5]}")
    
    mlflow.log_param("variance_threshold", variance_threshold)
    mlflow.log_param("strategy", "VarianceThreshold")
    mlflow.log_metric("low_variance_features", len(low_var_cols))
    mlflow.log_metric("kept_after_variance", len(num_cols) - len(low_var_cols))
    
    del num_data_filled
    gc.collect()

In [ ]:
# ============================================================
# RUN: RF_FS_HighCorrelation_FullData
# ============================================================
with mlflow.start_run(run_name="RF_FS_HighCorrelation_FullData"):
    
    correlation_threshold = 0.95
    
    print(f"Computing correlation on FULL dataset ({len(train_fe):,} rows)...")
    
    
    # მთლიან data-ზე
    full_data = train_fe[num_cols].fillna(train_fe[num_cols].median())
    
    print(f"Data shape: {full_data.shape}")
    print(f"Data memory: {full_data.memory_usage().sum() / 1024**2:.1f} MB")
    
    # numpy float32 - 2x ნაკლები RAM vidre float64
    data_array = full_data.values.astype(np.float32)
    del full_data
    gc.collect()
    
    print("Standardizing...")
    means = data_array.mean(axis=0)
    stds = data_array.std(axis=0) + 1e-8
    data_array = (data_array - means) / stds
    
    print("Computing correlation matrix (X.T @ X)...")
    n = data_array.shape[0]
    corr_matrix = np.abs((data_array.T @ data_array) / n).astype(np.float32)
    
    # data_array-ი აღარ გვჭირდება
    del data_array
    gc.collect()
    
    # ზედა სამკუთხა
    np.fill_diagonal(corr_matrix, 0)
    upper_tri = np.triu(corr_matrix, k=1)
    
    # high correlation pairs
    high_corr_indices = np.where(upper_tri > correlation_threshold)
    
    cols_array = np.array(num_cols)
    high_corr_cols_set = set(cols_array[high_corr_indices[0]]) | set(cols_array[high_corr_indices[1]])
    high_corr_cols = list(high_corr_cols_set)
    
    # წყვილები
    pairs_data = []
    for i, j in zip(high_corr_indices[0], high_corr_indices[1]):
        pairs_data.append((cols_array[i], cols_array[j], float(upper_tri[i, j])))
    pairs_data.sort(key=lambda x: x[2], reverse=True)
    
    print(f"\n Done!")
    print(f"\nმაღალი კორელაციის (>{correlation_threshold}) სვეტები: {len(high_corr_cols)}")
    print(f"მაღალი კორელაციის წყვილების რაოდენობა: {len(pairs_data)}")
    
    print(f"\n Top 20 ყველაზე მაღალი კორელაციის წყვილი:\n")
    print(f"{'Feature 1':<20} {'Feature 2':<20} {'Corr':<8}")
    print("-" * 50)
    for f1, f2, corr in pairs_data[:20]:
        print(f"{f1:<20} {f2:<20} {corr:.4f}")
    
    # ─────────────────────────────────────────────────────────
    # Heatmap (Top 40)
    # ─────────────────────────────────────────────────────────
    if len(high_corr_cols) > 0:
        viz_n = min(40, len(high_corr_cols))
        viz_cols = high_corr_cols[:viz_n]
        viz_indices = [num_cols.index(c) for c in viz_cols]
        
        viz_corr = np.abs(corr_matrix[np.ix_(viz_indices, viz_indices)])
        np.fill_diagonal(viz_corr, 1.0)
        
        plt.figure(figsize=(14, 12))
        sns.heatmap(viz_corr, 
                    xticklabels=viz_cols, 
                    yticklabels=viz_cols,
                    cmap='coolwarm', center=0.5, vmin=0, vmax=1,
                    square=True, linewidths=0.3,
                    cbar_kws={"shrink": 0.8, "label": "|Correlation|"})
        plt.title(f'Top {viz_n} High-Correlation Features (Full Dataset)')
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()
    
    # MLflow logging
    mlflow.log_param("correlation_threshold", correlation_threshold)
    mlflow.log_param("strategy", "numpy_correlation_full_data")
    mlflow.log_param("sample_size_for_corr", len(train_fe))  # full data
    mlflow.log_metric("high_correlation_features", len(high_corr_cols))
    mlflow.log_metric("high_correlation_pairs", len(pairs_data))
    mlflow.log_metric("kept_after_correlation", len(num_cols) - len(high_corr_cols))
    
    # CSV artifact
    pairs_df = pd.DataFrame(pairs_data[:100], columns=['Feature_1', 'Feature_2', 'Correlation'])
    pairs_df.to_csv('/tmp/high_corr_pairs_full.csv', index=False)
    mlflow.log_artifact('/tmp/high_corr_pairs_full.csv')
    
    # cleanup
    del corr_matrix, upper_tri
    gc.collect()
    
    

In [ ]:
# ============================================================
# RUN: RF_FS_TargetCorrelation_FullData
# ============================================================
with mlflow.start_run(run_name="RF_FS_TargetCorrelation_FullData"):
    
    print(f"Computing target correlation on FULL dataset ({len(train_fe):,} rows)...")
    
    # მთლიანი data + target
    full_data = train_fe[num_cols + [TARGET]].fillna(train_fe[num_cols].median())
    
    # numpy float32
    data_array = full_data[num_cols].values.astype(np.float32)
    target_array = full_data[TARGET].values.astype(np.float32)
    
    del full_data
    gc.collect()
    
    print("Standardizing...")
    # Standardize features
    means = data_array.mean(axis=0)
    stds = data_array.std(axis=0) + 1e-8
    data_array = (data_array - means) / stds
    
    # Standardize target
    target_mean = target_array.mean()
    target_std = target_array.std() + 1e-8
    target_standardized = (target_array - target_mean) / target_std
    
    print("Computing correlations with target...")
    # Correlation = (1/n) * sum(X_i_standardized * y_standardized)
    n = data_array.shape[0]
    target_corr_values = np.abs((data_array.T @ target_standardized) / n)
    
    # Series-ად რომ დავალაგოთ
    target_corr = pd.Series(target_corr_values, index=num_cols).sort_values(ascending=False)
    
    target_corr_threshold = 0.01
    low_target_corr = target_corr[target_corr < target_corr_threshold].index.tolist()
    
    print(f"\n Done!")
    print(f"\nTarget-თან თითქმის ნულ-კორელაციის სვეტები (<{target_corr_threshold}): {len(low_target_corr)}")
    print(f"Max target correlation: {target_corr.max():.4f}")
    print(f"Mean target correlation: {target_corr.mean():.4f}")
    
    print(f"\n Top 15 ყველაზე მნიშვნელოვანი feature target-თან კორელაციით:\n")
    print(target_corr.head(15).to_string())
    
    print(f"\n Bottom 10 (ნული-კორელაციის) feature:\n")
    print(target_corr.tail(10).to_string())
    
    # ─────────────────────────────────────────────────────────
    # Plot - Top 25 feature target-თან კორელაციით
    # ─────────────────────────────────────────────────────────
    plt.figure(figsize=(10, 10))
    target_corr.head(25).plot(kind='barh', color='steelblue', edgecolor='black')
    plt.xlabel('|Correlation with isFraud|')
    plt.title('Top 25 Features by Target Correlation (Full Dataset)')
    plt.gca().invert_yaxis()
    plt.axvline(x=target_corr_threshold, color='red', linestyle='--', alpha=0.5, label=f'Threshold = {target_corr_threshold}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # ─────────────────────────────────────────────────────────
    # Plot - target correlation distribution
    # ─────────────────────────────────────────────────────────
    plt.figure(figsize=(12, 4))
    plt.hist(target_corr.values, bins=80, color='coral', edgecolor='black')
    plt.axvline(x=target_corr_threshold, color='red', linestyle='--', label=f'Threshold = {target_corr_threshold}')
    plt.xlabel('|Correlation with isFraud|')
    plt.ylabel('რაოდენობა')
    plt.title('Target Correlation Distribution Across Features')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # MLflow logging
    mlflow.log_param("target_corr_threshold", target_corr_threshold)
    mlflow.log_param("strategy", "numpy_correlation_full_data")
    mlflow.log_param("sample_size", len(train_fe))
    
    mlflow.log_metric("low_target_corr_features", len(low_target_corr))
    mlflow.log_metric("max_target_correlation", float(target_corr.max()))
    mlflow.log_metric("mean_target_correlation", float(target_corr.mean()))
    mlflow.log_metric("min_target_correlation", float(target_corr.min()))
    
    # CSV artifact - top 50 features
    target_corr_df = target_corr.head(50).reset_index()
    target_corr_df.columns = ['Feature', 'Target_Correlation']
    target_corr_df.to_csv('/tmp/top_target_corr.csv', index=False)
    mlflow.log_artifact('/tmp/top_target_corr.csv')
    
    # cleanup
    del data_array, target_array, target_standardized
    gc.collect()
    
    

In [ ]:
# ============================================================
# RUN: RF_FS_Final
# ============================================================
with mlflow.start_run(run_name="RF_FS_Final"):
    # Threshold ცვლადები (Restart-ის შემდეგ ხელახლა)
    VARIANCE_THRESHOLD = 0.01
    CORRELATION_THRESHOLD = 0.95
    TARGET_CORR_THRESHOLD = 0.01

    print(f"Thresholds დაყენდა:")
    print(f"  Variance: {VARIANCE_THRESHOLD}")
    print(f"  Correlation: {CORRELATION_THRESHOLD}")
    print(f"  Target correlation: {TARGET_CORR_THRESHOLD}")
    
    # ვაერთიანებთ ყველა "ცუდი" feature-ის list-ს (set union)
    cols_to_drop_fs = set(low_var_cols) | set(high_corr_cols) | set(low_target_corr)
    
    print("Feature Selection შედეგები:")
    print(f"  Low variance:           {len(low_var_cols):4d}")
    print(f"  High correlation:       {len(high_corr_cols):4d}")
    print(f"  Low target correlation: {len(low_target_corr):4d}")
    print(f"  ─────────────────────────────────")
    print(f"  სულ უნიკალური წასაშლელი: {len(cols_to_drop_fs):4d}")
    
    # რა numerical features დაგვრჩა
    selected_num_cols = [c for c in num_cols if c not in cols_to_drop_fs]
    
    # categorical-ი ისე რჩება — მათ ცალცალკე encoding-ით ვამუშავებთ Training-ში
    selected_features = selected_num_cols + cat_cols
    
    print(f"\n საბოლოო Features:")
    print(f"  Numerical:    {len(selected_num_cols)} (იყო {len(num_cols)})")
    print(f"  Categorical:  {len(cat_cols)}")
    print(f"  ─────────────────────")
    print(f"  სულ:           {len(selected_features)}")
    
    # Overlap analysis
    only_lowvar = set(low_var_cols) - set(high_corr_cols) - set(low_target_corr)
    only_highcorr = set(high_corr_cols) - set(low_var_cols) - set(low_target_corr)
    only_targetcorr = set(low_target_corr) - set(low_var_cols) - set(high_corr_cols)
    overlap_all = set(low_var_cols) & set(high_corr_cols) & set(low_target_corr)
    
    print(f"\n Overlap ანალიზი:")
    print(f"  მხოლოდ low variance:        {len(only_lowvar)}")
    print(f"  მხოლოდ high correlation:    {len(only_highcorr)}")
    print(f"  მხოლოდ low target corr:     {len(only_targetcorr)}")
    print(f"  ყველა სამში:                {len(overlap_all)}")
    
    # MLflow logging
    mlflow.log_param("strategy", "combined_lowvar_highcorr_lowtargetcorr")
    mlflow.log_param("variance_threshold", VARIANCE_THRESHOLD)
    mlflow.log_param("correlation_threshold", CORRELATION_THRESHOLD)
    mlflow.log_param("target_corr_threshold", TARGET_CORR_THRESHOLD)
    
    mlflow.log_metric("features_before_FS", len(num_cols) + len(cat_cols))
    mlflow.log_metric("features_after_FS", len(selected_features))
    mlflow.log_metric("dropped_total", len(cols_to_drop_fs))
    mlflow.log_metric("selected_numerical", len(selected_num_cols))
    mlflow.log_metric("selected_categorical", len(cat_cols))
    mlflow.log_metric("only_low_variance", len(only_lowvar))
    mlflow.log_metric("only_high_correlation", len(only_highcorr))
    mlflow.log_metric("only_low_target_corr", len(only_targetcorr))
    mlflow.log_metric("overlap_all_three", len(overlap_all))
    
    # Selected features artifact
    selected_features_df = pd.DataFrame({
        'feature': selected_features,
        'type': ['numerical' if c in selected_num_cols else 'categorical' for c in selected_features]
    })
    selected_features_df.to_csv('/tmp/selected_features.csv', index=False)
    mlflow.log_artifact('/tmp/selected_features.csv')
    
    # Dropped features artifact
    dropped_features_df = pd.DataFrame({
        'feature': list(cols_to_drop_fs),
        'reason': [
            ', '.join([
                'low_var' if c in low_var_cols else '',
                'high_corr' if c in high_corr_cols else '',
                'low_target_corr' if c in low_target_corr else ''
            ]).strip(', ').replace(', ,', ',')
            for c in cols_to_drop_fs
        ]
    })
    dropped_features_df.to_csv('/tmp/dropped_features.csv', index=False)
    mlflow.log_artifact('/tmp/dropped_features.csv')
    
    print(f"\n selected_num_cols და cat_cols მზადაა Training-ისთვის")
    

# 4. Training

ვაშენებთ Random Forest Pipeline-ს:
- **Preprocessing**: Imputation (NaN-ების შევსება) + Frequency Encoding (RF-ი scaling-ს არ საჭიროებს)
- **Model**: Random Forest Classifier
- **Validation**: Time-based 80/20 split
- **Hyperparameter tuning**: სხვადასხვა n_estimators, max_depth, min_samples_leaf-ით ვცდით
- **Final**: საუკეთესო Pipeline ჩაიწერება MLflow Model Registry-ში


In [ ]:
import gc

def reduce_mem_usage(df):
    """Memory reduction - downcasts numeric types."""
    start_mem = df.memory_usage().sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            try:
                c_min = df[col].min()
                c_max = df[col].max()
                
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                else:
                    if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
            except:
                pass
    
    end_mem = df.memory_usage().sum() / 1024**2
    reduction = 100 * (start_mem - end_mem) / start_mem
    print(f'  {start_mem:.1f} MB → {end_mem:.1f} MB ({reduction:.1f}% reduction)')
    return df

print("Reducing train_fe memory...")
train_fe = reduce_mem_usage(train_fe)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'category_encoders', '-q'])

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin
from category_encoders import CountEncoder
import time




In [ ]:
print("Categorical features cardinality:\n")
print(f"{'Feature':<25} {'Unique values':<15}")
print("-" * 45)

cardinality_data = []
for col in cat_cols:
    n_unique = train_fe[col].nunique()
    cardinality_data.append((col, n_unique))

cardinality_data.sort(key=lambda x: x[1], reverse=True)

for col, n in cardinality_data:
    print(f"{col:<25} {n:<15}")

In [ ]:
# ─────────────────────────────────────────────────────────
# Custom string imputer
# ─────────────────────────────────────────────────────────
class StringImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        return X.astype(str).fillna('missing').replace('nan', 'missing')

# ─────────────────────────────────────────────────────────
# RF-ისთვის გამარტივებული Pipeline
# ─────────────────────────────────────────────────────────
# რატომ მარტივი?
# - RF tree-based - არ სჭირდება scaling (split threshold-ის მიხედვით)
# - Frequency Encoding cardinality-ის სიგნალს ინახავს
# - ერთი approach ყველა cat-ისთვის - სწრაფი, მცირე memory

# Numerical: მხოლოდ imputation
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical: Frequency Encoding ყველაფერზე
categorical_pipeline = Pipeline([
    ('string_imputer', StringImputer()),
    ('frequency', CountEncoder())
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, selected_num_cols),
    ('cat', categorical_pipeline, cat_cols)
], remainder='drop')

print(f" RF Preprocessor:")
print(f"  Numerical:    {len(selected_num_cols)} → Imputer (no scaler)")
print(f"  Categorical:  {len(cat_cols)} → Frequency Encoding")


In [ ]:
TARGET = 'isFraud'

print("Preparing X, y...")
df_sorted = train_fe.sort_values('TransactionDT').reset_index(drop=True)
X = df_sorted[selected_num_cols + cat_cols]
y = df_sorted[TARGET]

print(f"\nReducing X memory...")
X = reduce_mem_usage(X)

print(f"\n X shape: {X.shape}")
print(f"  X memory: {X.memory_usage().sum() / 1024**2:.1f} MB")
print(f"  Fraud rate: {y.mean():.4f}")

del df_sorted
gc.collect()

In [ ]:
# ============================================================
# RF Hyperparameter Comparison: 5 ცდა ცალცალკე MLflow run-ებად
# ============================================================
import time
from sklearn.metrics import roc_auc_score

# Time-based split
split_idx = int(len(X) * 0.8)
X_train = X.iloc[:split_idx]
y_train = y.iloc[:split_idx]
X_val = X.iloc[split_idx:]
y_val = y.iloc[split_idx:]

print(f"Split: Train {X_train.shape}, Val {X_val.shape}")

# ─────────────────────────────────────────────────────────
# Sample-ზე ვცდით (RF სრულ data-ზე ნელია)
# ─────────────────────────────────────────────────────────
SAMPLE_SIZE = 200_000
if len(X_train) > SAMPLE_SIZE:
    X_train_sample = X_train.iloc[:SAMPLE_SIZE]
    y_train_sample = y_train.iloc[:SAMPLE_SIZE]
    print(f"Using sample of {SAMPLE_SIZE:,} rows for hyperparameter search")
else:
    X_train_sample = X_train
    y_train_sample = y_train

# ─────────────────────────────────────────────────────────
# 5 RF configurations
# ─────────────────────────────────────────────────────────
configs = [
    {
        'name': 'RF_Baseline',
        'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 100,
        'description': 'Baseline - small trees'
    },
    {
        'name': 'RF_DeepTrees',
        'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 100,
        'description': 'Deeper trees - capture complex patterns'
    },
    {
        'name': 'RF_MoreTrees',
        'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 50,
        'description': 'More trees - lower variance'
    },
    {
        'name': 'RF_LowMinSamples',
        'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 70,
        'description': 'Allow leaf with single sample (overfit risk)'
    },
    {
        'name': 'RF_HighMinSamples',
        'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 50,
        'description': 'Conservative leafs - regularization'
    },
]

results = []

for cfg in configs:
    with mlflow.start_run(run_name=cfg['name']):
        
        print(f"\n{'='*55}")
        print(f"{cfg['name']}")
        print(f"  {cfg['description']}")
        print(f"  n_est={cfg['n_estimators']}, max_depth={cfg['max_depth']}, min_leaf={cfg['min_samples_leaf']}")
        print(f"{'='*55}")
        
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', RandomForestClassifier(
                n_estimators=cfg['n_estimators'],
                max_depth=cfg['max_depth'],
                min_samples_leaf=cfg['min_samples_leaf'],
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            ))
        ])
        
        start = time.time()
        pipeline.fit(X_train_sample, y_train_sample)
        train_time = time.time() - start
        
        val_pred = pipeline.predict_proba(X_val)[:, 1]
        train_pred = pipeline.predict_proba(X_train_sample)[:, 1]
        
        val_auc = roc_auc_score(y_val, val_pred)
        train_auc = roc_auc_score(y_train_sample, train_pred)
        gap = train_auc - val_auc
        
        if gap > 0.10:
            analysis = "Strong overfitting"
        elif gap > 0.05:
            analysis = "Moderate overfitting"
        elif gap < 0.02:
            analysis = "Possible underfitting"
        else:
            analysis = "Healthy gap"
        
        print(f"  Train AUC: {train_auc:.4f}")
        print(f"  Val AUC:   {val_auc:.4f}")
        print(f"  Gap:       {gap:.4f}  ({analysis})")
        print(f"  Time:      {train_time:.1f}s")
        
        # MLflow
        mlflow.log_param("n_estimators", cfg['n_estimators'])
        mlflow.log_param("max_depth", cfg['max_depth'])
        mlflow.log_param("min_samples_leaf", cfg['min_samples_leaf'])
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("description", cfg['description'])
        mlflow.log_param("validation", "TimeBased_80_20")
        mlflow.log_param("sample_size", len(X_train_sample))
        
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("overfit_gap", gap)
        mlflow.log_metric("training_time_sec", train_time)
        
        mlflow.set_tag("overfit_analysis", analysis)
        
        results.append({
            'name': cfg['name'],
            'n_estimators': cfg['n_estimators'],
            'max_depth': cfg['max_depth'],
            'min_samples_leaf': cfg['min_samples_leaf'],
            'train_auc': train_auc,
            'val_auc': val_auc,
            'gap': gap,
            'time_sec': train_time
        })

# Summary
print("\n" + "="*100)
print(" SUMMARY:")
print("="*100)

results_df = pd.DataFrame(results).sort_values('val_auc', ascending=False)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
best_n_est = int(best['n_estimators'])
best_max_depth = int(best['max_depth'])
best_min_leaf = int(best['min_samples_leaf'])

print(f"\n Best: {best['name']}")
print(f"   Val AUC: {best['val_auc']:.4f}")
print(f"   Config: n_est={best_n_est}, max_depth={best_max_depth}, min_leaf={best_min_leaf}")


In [ ]:
# ============================================================
# RUN: RF_Final_Model
# ============================================================
with mlflow.start_run(run_name="RF_Final_Model"):
    
    print(f"Training final RF on FULL data...")
    print(f"  n_est={best_n_est}, max_depth={best_max_depth}, min_leaf={best_min_leaf}")
    
    final_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=best_n_est,
            max_depth=best_max_depth,
            min_samples_leaf=best_min_leaf,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    print("Fitting on full data...")
    start = time.time()
    final_pipeline.fit(X, y)
    train_time = time.time() - start
    print(f" Trained in {train_time:.1f}s ({train_time/60:.1f} min)")
    
    mlflow.log_param("best_n_estimators", best_n_est)
    mlflow.log_param("best_max_depth", best_max_depth)
    mlflow.log_param("best_min_samples_leaf", best_min_leaf)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("trained_on", "full_data")
    mlflow.log_param("preprocessor_strategy", "Frequency_Encoding_only")
    mlflow.log_metric("val_auc_from_split", best['val_auc'])
    mlflow.log_metric("training_time_sec", train_time)
    mlflow.log_metric("n_train_rows", len(X))
    mlflow.log_metric("n_features", len(selected_num_cols) + len(cat_cols))
    
    mlflow.set_tag("model_type", "RandomForest")
    
    print("\nSaving Pipeline to Model Registry...")
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="rf_pipeline",
        registered_model_name="RF_Fraud_Detection"
    )
    


In [ ]:
print("Loading test data...")

test_tx = pd.read_csv(DATA_PATH + 'test_transaction.csv')
test_id = pd.read_csv(DATA_PATH + 'test_identity.csv')
test_id.columns = [c.replace('-', '_') for c in test_id.columns]

test = test_tx.merge(test_id, on='TransactionID', how='left')
print(f"Test shape: {test.shape}")

del test_tx, test_id
gc.collect()

# Memory reduce test
test = reduce_mem_usage(test)

# ─────────────────────────────────────────────────────────
# Test-ზე იგივე FE რაც train-ზე
# ─────────────────────────────────────────────────────────
print("\nApplying Feature Engineering...")
test['TransactionAmt_log'] = np.log1p(test['TransactionAmt'])
test['TransactionAmt_decimal'] = test['TransactionAmt'] - test['TransactionAmt'].astype(int)
test['hour'] = (test['TransactionDT'] / 3600) % 24
test['day'] = (test['TransactionDT'] / (3600 * 24)) % 7

for col in ['P_emaildomain', 'R_emaildomain']:
    if col in test.columns:
        test[col + '_bin'] = test[col].astype(str).str.split('.').str[0]

# ─────────────────────────────────────────────────────────
# X_test
# ─────────────────────────────────────────────────────────
X_test = test[selected_num_cols + cat_cols]
print(f"X_test shape: {X_test.shape}")

# ─────────────────────────────────────────────────────────
# Predict
# ─────────────────────────────────────────────────────────
print("\nPredicting...")
test_pred = final_pipeline.predict_proba(X_test)[:, 1]

# ─────────────────────────────────────────────────────────
# Submission file
# ─────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'TransactionID': test['TransactionID'],
    'isFraud': test_pred
})

submission_path = '/kaggle/working/submission_rf.csv'
submission.to_csv(submission_path, index=False)

print(f"\n Submission saved: {submission_path}")
print(f"  Rows: {len(submission):,}")
print(f"\nPrediction distribution:")
print(submission['isFraud'].describe())

# Histogram
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.hist(submission['isFraud'], bins=100, color='steelblue', edgecolor='black')
plt.title('RF Test Predictions Distribution')
plt.xlabel('Predicted P(fraud)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
